# Web Search RAG with Tavily

This notebook demonstrates how to build a RAG system that uses **real-time web search** as its knowledge source instead of static documents. We'll use Tavily - a search API built specifically for LLMs and RAG applications.

## What is Tavily?

Tavily is a search engine optimized for AI agents and RAG workflows. Unlike traditional search engines that return HTML pages, Tavily returns:
- Clean, structured text content
- Relevance scores
- Source citations
- LLM-ready context

## Why Web Search RAG?

| Traditional RAG | Web Search RAG |
|-----------------|----------------|
| Static documents | Real-time data |
| Knowledge cutoff date | Always current |
| Limited to your corpus | Access to entire web |
| Requires re-indexing | No indexing needed |
| Good for internal docs | Good for current events |

## What We'll Build

```
User Question ──► Tavily Search ──► Retrieved Context ──► LLM ──► Answer
                     │                    │
                     ▼                    ▼
              Real-time web          Fresh, relevant
              search results         information
```

Tavily, Parallel AI, Exa, Linkup

## 1. Environment Setup

First, let's install the required packages and set up our API keys.

In [ ]:
# Install required packages
!pip install -q tavily-python openai python-dotenv

In [ ]:
import os
from getpass import getpass

# Set up API keys securely
# You can get a free Tavily API key at: https://tavily.com/
os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API key: ")
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("API keys set successfully!")

Enter your Tavily API key: ··········
Enter your OpenAI API key: ··········
API keys set successfully!


In [ ]:
from tavily import TavilyClient
from openai import OpenAI
import json

# Initialize clients
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
openai_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("Clients initialized successfully!")

Clients initialized successfully!


## 2. Understanding Tavily Search Methods

Tavily provides three main methods for different use cases:

| Method | Use Case | Returns |
|--------|----------|--------|
| `search()` | Full control over search | Complete response with all metadata |
| `get_search_context()` | RAG applications | Clean context string ready for LLM |
| `qna_search()` | Quick answers | Direct answer from Tavily's LLM |

Let's explore each one.

### 2. Basic Search - Full Response

The `search()` method gives you complete control and returns detailed results.

In [ ]:
# Basic search example
query = "What are the latest AI models that got released recently in the past few days in February 2026?"

response = tavily_client.search(
    query=query,
    search_depth="basic",  # "basic" or "advanced" (advanced is more thorough)
    max_results=6,         # Number of results (0-20)
    include_answer=True,   # Include an LLM-generated answer
)

print("=" * 60)
print(f"Query: {query}")
print("=" * 60)

# Show the generated answer
if response.get("answer"):
    print(f"\nGenerated Answer:\n{response['answer']}")

# Show the search results
print(f"\n{'='*60}")
print(f"Found {len(response['results'])} results:")
print("=" * 60)

for i, result in enumerate(response["results"], 1):
    print(f"\n{i}. {result['title']}")
    print(f"   URL: {result['url']}")
    print(f"   Score: {result.get('score', 'N/A')}")
    print(f"   Content: {result['content'][:200]}...")

Query: What are the latest AI models that got released recently in the past few days in February 2026?

Generated Answer:
In February 2026, notable AI models released include GPT-5.2 and its variants, Qwen 3.5 by Alibaba, and Gemini 3 Pro. Google introduced Nano Banana 2 and Deep Think. Major models like GPT-5.3 and Grok 4.20 were also launched.

Found 6 results:

1. New AI Model Releases News | February, 2026 (STARTUPS EDITION)
   URL: https://blog.mean.ceo/new-ai-model-releases-news-february-2026/
   Score: 0.94620544
   Content: Recent releases in AI models include GPT-5.2, optimized for coding and industry tasks, along with its smaller versions GPT-5 mini and GPT-5 nano...

2. Every AI Model Released in February 2026 - Adam Holter
   URL: https://adam.holter.com/every-ai-model-released-in-february-2026/
   Score: 0.92620385
   Content: Every AI Model Released in February 2026 · Google DeepMind · Anthropic · OpenAI · xAI · Zhipu AI · Alibaba · ByteDance and Kuaishou · MiniMax....

3

### Understanding the Response Structure

```python
{
    "query": "your search query",
    "answer": "LLM-generated answer (if include_answer=True)",
    "results": [
        {
            "title": "Page title",
            "url": "https://...",
            "content": "Extracted text content",
            "score": 0.95,  # Relevance score
            "raw_content": "Full page content (if requested)"
        },
        ...
    ]
}
```

## 3. Advanced Search Options

Tavily provides powerful filtering and customization options.

In [ ]:
# Advanced search with filters
response = tavily_client.search(
    query="OpenAI GPT-5 announcements",

    # Search depth
    search_depth="advanced",  # More thorough but slower

    # Topic specialization
    topic="news",  # "general", "news", or "finance"

    # Time filtering
    time_range="week",  # "day", "week", "month", "year"

    # Domain filtering
    include_domains=["openai.com", "techcrunch.com", "theverge.com"],
    # exclude_domains=["reddit.com"],  # Exclude specific domains

    # Results
    max_results=10,
    include_answer=True,
)

print(f"Answer: {response.get('answer', 'No answer generated')}")
print(f"\nSources:")
for result in response["results"]:
    print(f"  - {result['title'][:60]}... ({result['url'][:50]}...)")

Answer: Based on the most recent information available, there have been no official announcements regarding the release of GPT-5 by OpenAI. The company has been focusing on other initiatives, such as enhancing safety measures for its AI models and collaborating with organizations to develop tools for teen safety. Additionally, there have been strategic shifts and discontinuations of certain features and projects, indicating a reevaluation of priorities. For the latest updates, it is recommended to refer to official communications from OpenAI.

Sources:
  - OpenAI adds open source tools to help developers build for t... (https://techcrunch.com/2026/03/24/openai-adds-open...)
  - OpenAI abandons yet another side quest: ChatGPT’s erotic mod... (https://techcrunch.com/2026/03/26/openai-abandons-...)
  - OpenAI shelves erotic chatbot ‘indefinitely’ - The Verge... (https://www.theverge.com/ai-artificial-intelligenc...)
  - Why SoftBank’s new $40B loan points to a 2026 OpenAI IPO - T... (http

## 4. Building a Web Search RAG System

Now let's build a complete RAG system that:
1. Takes a user question
2. Searches the web for relevant context using Tavily
3. Passes the context to an LLM (OpenAI)
4. Returns a grounded answer with sources

### Architecture

```
┌─────────────────────────────────────────────────────────────────────┐
│                    WEB SEARCH RAG PIPELINE                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   ┌──────────┐    ┌──────────────┐    ┌─────────┐    ┌──────────┐ │
│   │   User   │    │   Tavily     │    │  Build  │    │   LLM    │ │
│   │ Question │───►│   Search     │───►│  Prompt │───►│  (GPT)   │ │
│   └──────────┘    └──────────────┘    └─────────┘    └────┬─────┘ │
│                          │                                │       │
│                          ▼                                ▼       │
│                   Real-time web                    Grounded       │
│                   search results                   Answer +       │
│                   with sources                     Citations      │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
def web_search_rag(question: str, max_results: int = 5, search_depth: str = "advanced") -> dict:
    """
    Perform RAG using real-time web search.

    Args:
        question: The user's question
        max_results: Number of web results to retrieve
        search_depth: "basic" or "advanced"

    Returns:
        Dictionary with answer, sources, and metadata
    """

    # Step 1: Search the web for relevant context
    print(f"Searching the web for: '{question}'")

    search_response = tavily_client.search(
        query=question,
        search_depth=search_depth,
        max_results=max_results,
        include_answer=False,  # We'll generate our own answer
    )

    # Step 2: Extract and format the context
    sources = []
    context_parts = []

    for i, result in enumerate(search_response["results"], 1):
        # Build context from each result
        context_parts.append(f"[Source {i}]: {result['content']}")

        # Store source info for citation
        sources.append({
            "index": i,
            "title": result["title"],
            "url": result["url"],
            "score": result.get("score", 0)
        })

    # Combine all context
    full_context = "\n\n".join(context_parts)
    print(f"Retrieved {len(sources)} sources from the web")

    # Step 3: Build the prompt for the LLM
    system_prompt = """You are a helpful assistant that answers questions based on the provided web search results.

Guidelines:
- Base your answer ONLY on the provided context
- Cite sources using [Source N] notation
- If the context doesn't contain enough information, say so
- Be concise but comprehensive
- Include relevant details and data when available"""

    user_prompt = f"""Based on the following web search results, answer this question: {question}

Web Search Results:
{full_context}

Please provide a comprehensive answer with source citations."""

    # Step 4: Generate answer using LLM
    print("Generating answer with LLM...")

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",  # or "gpt-4o" for better quality
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,  # Low temperature for factual responses
        max_tokens=1000
    )

    answer = response.choices[0].message.content

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context_length": len(full_context),
        "model": "gpt-4o-mini"
    }

In [ ]:
def display_rag_result(result: dict):
    """Display RAG results in a formatted way."""

    print("=" * 70)
    print(f"QUESTION: {result['question']}")
    print("=" * 70)

    print(f"\nANSWER:\n{result['answer']}")

    print(f"\n{'='*70}")
    print("SOURCES:")
    print("-" * 70)

    for source in result["sources"]:
        print(f"[{source['index']}] {source['title']}")
        print(f"    URL: {source['url']}")
        print(f"    Relevance: {source['score']:.2f}")
        print()

    print(f"Context size: {result['context_length']} characters")
    print(f"Model: {result['model']}")

### Let's Test Our Web Search RAG System!

In [ ]:
# Test with a current events question
question = "What are the latest developments in AI regulation in the European Union?"

result = web_search_rag(question)
display_rag_result(result)

Searching the web for: 'What are the latest developments in AI regulation in the European Union?'
Retrieved 5 sources from the web
Generating answer with LLM...
QUESTION: What are the latest developments in AI regulation in the European Union?

ANSWER:
The latest developments in AI regulation within the European Union (EU) include significant adjustments to the EU AI Act and ongoing discussions about a digital omnibus regulation. Here are the key points:

1. **EU AI Act Implementation**: The EU AI Act, which is the first comprehensive legal framework for AI globally, is set to begin taking effect in phases starting in 2025. High-risk AI systems will face strict obligations, with rules coming into effect in August 2026 and August 2027. The Act aims to foster trustworthy AI while addressing various risks associated with AI technologies [Source 3].

2. **Digital Omnibus Regulation Proposal**: EU lawmakers are considering a digital omnibus regulation that may delay certain obligations rela

In [ ]:
# Test with a technical question
question = "How does Claude Opus 4.6 Sonnet compare to GPT-5.3-Codex in coding benchmarks?"

result = web_search_rag(question)
display_rag_result(result)

Searching the web for: 'How does Claude Opus 4.6 Sonnet compare to GPT-5.3-Codex in coding benchmarks?'
Retrieved 5 sources from the web
Generating answer with LLM...
QUESTION: How does Claude Opus 4.6 Sonnet compare to GPT-5.3-Codex in coding benchmarks?

ANSWER:
Claude Opus 4.6 and GPT-5.3-Codex exhibit distinct strengths in coding benchmarks, reflecting their different design focuses. Here’s a detailed comparison based on the available data:

### Performance Overview
1. **Reasoning and Knowledge Work**:
   - **Claude Opus 4.6** excels in reasoning-heavy benchmarks and complex knowledge tasks. It scored **85.1%** on the MMLU Pro benchmark and **77.3%** on GPQA Diamond, outperforming GPT-5.3-Codex in these areas [Source 1][Source 3].
   - **GPT-5.3-Codex**, while strong in coding-specific tasks, scored lower on these reasoning benchmarks, with **82.9%** on MMLU Pro and **73.8%** on GPQA Diamond [Source 1].

2. **Coding-Specific Benchmarks**:
   - In coding benchmarks, **Claude Opus 4.

## 5. Enhanced RAG with Topic-Specific Search

Tavily supports different search topics for optimized results:
- `general` - Default, broad web search
- `news` - Recent news articles
- `finance` - Financial data and analysis

In [ ]:
def topic_specific_rag(question: str, topic: str = "general") -> dict:
    """
    RAG with topic-specific search optimization.

    Args:
        question: The user's question
        topic: "general", "news", or "finance"
    """

    print(f"Topic: {topic.upper()}")
    print(f"Question: {question}")
    print("-" * 50)

    # Search with topic specialization
    search_response = tavily_client.search(
        query=question,
        topic=topic,
        search_depth="advanced",
        max_results=5,
        time_range="week" if topic == "news" else None,  # Recent for news
    )

    # Format context
    context = "\n\n".join([
        f"[{i}] {r['content']}"
        for i, r in enumerate(search_response["results"], 1)
    ])

    # Topic-specific system prompts
    topic_prompts = {
        "general": "You are a helpful assistant that provides accurate information based on web search results.",
        "news": "You are a news analyst. Summarize recent developments and cite sources with dates when available.",
        "finance": "You are a financial analyst. Provide data-driven insights and always mention relevant numbers, percentages, and trends."
    }

    # Generate answer
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": topic_prompts[topic]},
            {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"}
        ],
        temperature=0.1
    )

    print(f"\nAnswer:\n{response.choices[0].message.content}")

    print(f"\nSources:")
    for i, r in enumerate(search_response["results"], 1):
        print(f"  [{i}] {r['title'][:50]}... - {r['url'][:40]}...")

    return response.choices[0].message.content

In [ ]:
# News-specific search
news_answer = topic_specific_rag(
    "What are the latest AI announcements from major tech companies this week?",
    topic="news"
)

In [ ]:
# Finance-specific search
finance_answer = topic_specific_rag(
    "How is NVIDIA stock performing and what are analysts saying?",
    topic="finance"
)

## 6. Simplified RAG with get_search_context()

For the simplest implementation, use `get_search_context()` which handles all the context extraction for you.

In [ ]:
def simple_web_rag(question: str) -> str:
    """
    Simplest possible Web Search RAG implementation.
    Uses get_search_context() for automatic context handling.
    """

    # Step 1: Get search context (Tavily handles everything)
    context = tavily_client.get_search_context(
        query=question,
        max_results=5
    )

    # Step 2: Send to LLM
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Answer based on the provided context. Be concise and accurate."
            },
            {
                "role": "user",
                "content": f"Context: {context}\n\nQuestion: {question}"
            }
        ],
        temperature=0.1
    )

    return response.choices[0].message.content


# Test it!
question = "What is LangChain and what are its main features?"
answer = simple_web_rag(question)

print(f"Q: {question}")
print(f"\nA: {answer}")

## 7. Interactive Q&A Interface

Let's create an interactive interface for asking questions.

In [ ]:
def interactive_web_rag():
    """
    Interactive Q&A interface with web search RAG.
    """
    print("=" * 60)
    print("Web Search RAG - Ask anything about current events!")
    print("=" * 60)
    print("Type 'quit' to exit\n")

    while True:
        question = input("\nYour question: ").strip()

        if question.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        if not question:
            print("Please enter a question.")
            continue

        print("\nSearching and generating answer...")

        try:
            result = web_search_rag(question, max_results=5)

            print(f"\n{'─' * 60}")
            print(f"Answer:\n{result['answer']}")
            print(f"\n{'─' * 60}")
            print(f"Sources ({len(result['sources'])}):")
            for s in result['sources'][:3]:
                print(f"  • {s['title'][:50]}...")

        except Exception as e:
            print(f"Error: {e}")

# Uncomment to run interactively:
# interactive_web_rag()

## 8. Comparison: Traditional RAG vs Web Search RAG

Let's visualize the architectural differences.

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         TRADITIONAL RAG                                     │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌─────────┐    ┌─────────┐    ┌──────────┐    ┌─────────┐                 │
│  │Documents│───►│ Chunker │───►│ Embedder │───►│ Vector  │                 │
│  │ (PDFs)  │    │         │    │          │    │   DB    │                 │
│  └─────────┘    └─────────┘    └──────────┘    └────┬────┘                 │
│                                                     │                      │
│                                             (Indexed once)                 │
│                                                     │                      │
│  ┌─────────┐    ┌─────────┐    ┌──────────┐    ┌────▼────┐    ┌─────────┐ │
│  │  User   │───►│ Embed   │───►│ Retrieve │───►│ Context │───►│   LLM   │ │
│  │ Query   │    │ Query   │    │ Top-K    │    │         │    │         │ │
│  └─────────┘    └─────────┘    └──────────┘    └─────────┘    └─────────┘ │
│                                                                             │
│  Pros: Fast retrieval, works offline, full control over data              │
│  Cons: Static data, requires re-indexing, limited to your corpus          │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────────────────┐
│                         WEB SEARCH RAG                                      │
├─────────────────────────────────────────────────────────────────────────────┤
│                                                                             │
│  ┌─────────┐    ┌──────────────┐    ┌──────────┐    ┌─────────┐           │
│  │  User   │───►│   Tavily     │───►│ Retrieved│───►│   LLM   │           │
│  │ Query   │    │  Web Search  │    │  Context │    │         │           │
│  └─────────┘    └──────────────┘    └──────────┘    └─────────┘           │
│                        │                                                    │
│                        ▼                                                    │
│                 ┌──────────────┐                                           │
│                 │  Live Web    │                                           │
│                 │  (Billions   │                                           │
│                 │   of pages)  │                                           │
│                 └──────────────┘                                           │
│                                                                             │
│  Pros: Always current, no indexing needed, access to entire web           │
│  Cons: API costs, depends on internet, less control over sources          │
│                                                                             │
└─────────────────────────────────────────────────────────────────────────────┘
```

## 9. Best Practices and Tips

### When to Use Web Search RAG

| Use Case | Recommendation |
|----------|---------------|
| Current events, news | Web Search RAG |
| Company internal docs | Traditional RAG |
| Real-time data (stocks, weather) | Web Search RAG |
| Historical research papers | Traditional RAG |
| Product information | Hybrid (both) |
| Customer support | Hybrid (both) |

### Optimization Tips

1. **Use appropriate search depth**: `basic` for speed, `advanced` for accuracy
2. **Filter domains**: Use `include_domains` for trusted sources
3. **Time filtering**: Use `time_range` for recent information
4. **Topic selection**: Use `topic="news"` or `topic="finance"` for specialized searches
5. **Handle failures gracefully**: Always wrap in try-except

In [ ]:
# Production-ready example with error handling
def production_web_rag(question: str, fallback_answer: str = "I couldn't find relevant information.") -> dict:
    """
    Production-ready Web Search RAG with error handling.
    """
    try:
        # Search with timeout
        search_response = tavily_client.search(
            query=question,
            search_depth="advanced",
            max_results=5,
            timeout=30  # 30 second timeout
        )

        if not search_response.get("results"):
            return {"answer": fallback_answer, "sources": [], "error": None}

        # Build context
        context = "\n\n".join([
            f"[{i}] {r['content']}"
            for i, r in enumerate(search_response["results"], 1)
        ])

        # Generate with LLM
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Answer based on context. Cite sources as [N]."},
                {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"}
            ],
            temperature=0.1,
            timeout=30
        )

        return {
            "answer": response.choices[0].message.content,
            "sources": [r["url"] for r in search_response["results"]],
            "error": None
        }

    except Exception as e:
        return {
            "answer": fallback_answer,
            "sources": [],
            "error": str(e)
        }


# Test production version
result = production_web_rag("What is the current status of GPT-5 development?")
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources'][:3]}")
if result['error']:
    print(f"Error: {result['error']}")

## Conclusion

You've learned how to build a Web Search RAG system using Tavily! Here's what we covered:

### Key Takeaways

1. **Tavily Methods**:
   - `search()` - Full control, detailed results
   - `get_search_context()` - Optimized for RAG, returns clean context
   - `qna_search()` - Direct answers without your own LLM

2. **Architecture**:
   - Query → Web Search → Context → LLM → Answer
   - No indexing or embedding required
   - Always returns current information

3. **Best Practices**:
   - Use topic-specific searches for better results
   - Filter domains for trusted sources
   - Handle errors gracefully in production

### Next Steps

1. **Hybrid RAG**: Combine with traditional RAG for best of both worlds
2. **Caching**: Cache frequent queries to reduce API calls
3. **Streaming**: Use streaming for real-time response generation
4. **Multi-turn**: Build conversational interfaces with context memory

### Resources

- [Tavily Documentation](https://docs.tavily.com/)
- [Tavily Python SDK](https://github.com/tavily-ai/tavily-python)
- [Tavily Blog - Hybrid RAG](https://blog.tavily.com/hybrid-rag-with-tavily-combining-static-knowledge-and-dynamic-web-data/)

# Without Structured Extraction (The Problem)                                                                       

In [ ]:
CONFIG = {
    "llm": {
        "model": "gpt-4o",                    # OpenRouter model to use
        "temperature": 0.1                   # Temperature for response generation
    },
    "embeddings": {
        "model": "local:BAAI/bge-small-en-v1.5",  # Local embedding model (no API key needed)
        "chunk_size": 512,                 # Size of text chunks for processing
        "chunk_overlap": 50                 # Overlap between consecutive chunks
    },
    "vector_store": {
        "type": "lancedb",                   # Vector database type
        "table_name": "academic_papers",     # Table name for storing embeddings
        "path": "/content/storage/papers_vectordb"    # Path to vector database
    },
    "index": {
        "storage_path": "/content/storage/papers_index",  # Path to store complete index
        "similarity_top_k": 5                             # Number of similar chunks to retrieve
    },
    "papers": {
        "folder": "/content/drive/MyDrive/outskill_c4/papers/agents"    # Path to academic papers folder
    }
}

In [ ]:
CONFIG['llm']['model']

'gpt-4o'

In [ ]:
from openai import OpenAI

client = OpenAI()

# Raw event descriptions from different sources
event_text = """
Hey! We're hosting a Tech Meetup at the Bangalore Innovation Hub on March 15th, 2025.
It starts at 6:30 PM and we're expecting around 150 people. Entry is free but registration
is required. Contact Priya at priya@techmeetup.com for more info.
"""

# Ask LLM to extract event info
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract event details from the text."},
        {"role": "user", "content": event_text}
    ]
)

print(response.choices[0].message.content)

Event Details:
- **Event Name:** Tech Meetup
- **Date:** March 15th, 2025
- **Time:** 6:30 PM
- **Location:** Bangalore Innovation Hub
- **Expected Attendance:** Around 150 people
- **Entry Fee:** Free (registration required)
- **Contact for More Info:** Priya (email: priya@techmeetup.com)


In [ ]:
# Ask LLM to extract event info
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract event details from the text."},
        {"role": "user", "content": event_text}
    ]
)

print(response.choices[0].message.content)

Event Details:
- **Event Name**: Tech Meetup
- **Location**: Bangalore Innovation Hub
- **Date**: March 15th, 2025
- **Time**: 6:30 PM
- **Expected Attendance**: Around 150 people
- **Entry Fee**: Free (registration required)
- **Contact**: Priya at priya@techmeetup.com for more information.


In [ ]:
# Ask LLM to extract event info
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract event details from the text."},
        {"role": "user", "content": event_text}
    ]
)

print(response.choices[0].message.content)

Event Details:
- **Event:** Tech Meetup
- **Date:** March 15th, 2025
- **Time:** 6:30 PM
- **Location:** Bangalore Innovation Hub
- **Expected Attendance:** 150 people
- **Entry:** Free (registration required)
- **Contact for more info:** Priya at priya@techmeetup.com


# With Structured Extraction

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from datetime import date, time

# Through Pydantic we define exactly what we need
class EventInfo(BaseModel):
    """Structured event information for database storage."""

    event_name: str = Field(description="Name of the event")
    city: str = Field(description="City where event is held")
    venue: str = Field(description="Specific venue/location name")
    event_date: str = Field(description="Date in DD-MM-YYYY format")
    start_time: str = Field(description="Start time in 12 hour format (12-hour)")
    expected_attendees: int = Field(description="Expected number of participants")
    ticket_price: float = Field(description="Ticket price in INR, 0 if free")
    registration_required: bool = Field(description="Whether registration is required")
    organizer_name: Optional[str] = Field(default=None, description="Name of organizer if mentioned")
    organizer_email: Optional[str] = Field(default=None, description="Contact email if mentioned")

In [ ]:
# Step 2: Extract with Guaranteed Structure

from openai import OpenAI
import json

client = OpenAI()

event_text = """
Hey! We're hosting a Tech Meetup at the Bangalore Innovation Hub on March 15th, 2025. Ticket price is around 180 dollars.
It starts at 6:30 PM and we're expecting around 150 people. Entry is free but registration
is required. Contact Priya at priya@techmeetup.com for more info.
"""

# Pass Pydantic model directly to the LLM
response = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Extract event information from the text."},
        {"role": "user", "content": event_text}
    ],
    response_format=EventInfo  # ← Just pass the Pydantic model!
)
print(response)

# Already parsed as Pydantic object!
event_response = response.choices[0].message.parsed

# Parse JSON and validate with Pydantic
json_data = json.loads(response.choices[0].message.content)
print('--------------------------------------'*5)
print(json_data)

ParsedChatCompletion[TypeVar](id='chatcmpl-DOdblzqOCjRtIaaol6d4pYbLxiQ4E', choices=[ParsedChoice[TypeVar](finish_reason='stop', index=0, logprobs=None, message=ParsedChatCompletionMessage[TypeVar](content='{"event_name":"Tech Meetup","city":"Bangalore","venue":"Bangalore Innovation Hub","event_date":"15-03-2025","start_time":"6:30 PM","expected_attendees":150,"ticket_price":0,"registration_required":true,"organizer_name":"Priya","organizer_email":"priya@techmeetup.com"}', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None, parsed=EventInfo(event_name='Tech Meetup', city='Bangalore', venue='Bangalore Innovation Hub', event_date='15-03-2025', start_time='6:30 PM', expected_attendees=150, ticket_price=0.0, registration_required=True, organizer_name='Priya', organizer_email='priya@techmeetup.com')))], created=1774764497, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_e738e3044b', usage=Co

In [ ]:
json_data

{'event_name': 'Tech Meetup',
 'city': 'Bangalore',
 'venue': 'Bangalore Innovation Hub',
 'event_date': '15-03-2025',
 'start_time': '6:30 PM',
 'expected_attendees': 150,
 'ticket_price': 0,
 'registration_required': True,
 'organizer_name': 'Priya',
 'organizer_email': 'priya@techmeetup.com'}

In [ ]:
event_response

EventInfo(event_name='Tech Meetup', city='Bangalore', venue='Bangalore Innovation Hub', event_date='2025-03-15', start_time='18:30', expected_attendees=150, ticket_price=0.0, registration_required=True, organizer_name='Priya', organizer_email='priya@techmeetup.com')

In [ ]:
json_data = json.loads(response.choices[0].message.content)
json_data['city']

'Bangalore'

In [ ]:
# # Ask LLM to return JSON matching our schema
# response = client.chat.completions.create(
# model="gpt-4o-mini",
# messages=[
#     {
#         "role": "system",
#         "content": """Extract event information and return as JSON with these exact fields:
#         - event_name (string)
#         - city (string)
#         - venue (string)
#         - event_date (string, YYYY-MM-DD format)
#         - start_time (string, HH:MM 24-hour format)
#         - expected_attendees (integer)
#         - ticket_price (number, 0 if free)
#         - registration_required (boolean)
#         - organizer_name (string or null)
#         - organizer_email (string or null)

#         Return ONLY valid JSON, no other text."""
#     },
#     {"role": "user", "content": event_text}
# ],
# response_format={"type": "json_object"}  # Force JSON output
# )

In [ ]:
!pip install llama-index

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from llama_index.core.program import LLMTextCompletionProgram
from llama_index.core.output_parsers import PydanticOutputParser

class EventInfo(BaseModel):
    event_name: str = Field(description="Name of the event")
    city: str = Field(description="City where event is held")
    venue: str = Field(description="Specific venue/location name")
    event_date: str = Field(description="Date in YYYY-MM-DD format")
    start_time: str = Field(description="Start time in HH:MM 24-hour format")
    expected_attendees: int = Field(description="Expected number of participants")
    ticket_price: float = Field(description="Ticket price, 0 if free")
    registration_required: bool = Field(description="Whether registration is required")
    organizer_name: Optional[str] = Field(default=None, description="Organizer name")
    organizer_email: Optional[str] = Field(default=None, description="Contact email")


# Create extractor - Pydantic model is passed to the parser
event_extractor = LLMTextCompletionProgram.from_defaults(
    output_parser=PydanticOutputParser(EventInfo),  # ← Pass Pydantic model!
    prompt_template_str=(
        "Extract event information from this text:\n\n"
        "{text}\n\n"
        "Return the structured event data."
    )
)

# Use it
event = event_extractor(text=event_text)

print(f"Event: {event.event_name}")
print(f"City: {event.city}")

Event: Tech Meetup
City: Bangalore


In [ ]:
event

EventInfo(event_name='Tech Meetup', city='Bangalore', venue='Bangalore Innovation Hub', event_date='2025-03-15', start_time='18:30', expected_attendees=150, ticket_price=0.0, registration_required=True, organizer_name=None, organizer_email='priya@techmeetup.com')

## When to Use This                                                                                                                  
  1. **Use structured outputs when:**                                                                         
  - Building APIs that return event data                                                                          
  - Storing extracted data in databases                                                                     
  - Processing events from emails/messages/forms                                                                         
  - Building event aggregators across cities                                                                        
  - Any time you need reliable, parseable data                                                                                                                                                                                                 
  2. **Don't use when:**                                                                         
  - Just chatting with users about events                                                                        
  - Exploratory questions ("tell me about fun events")                                                                      
  - When format doesn't matter